## API Intro Classwork - pokeapi.co
Describe at least two endpoints that seem intriguing. What are the route parameters being used in each? What data does it retrieve? Include this as a Markdown block in your notebook. Pull the data from the API as a fixture first and practice importing it into your notebook as a JSON file. If you have time, practice achieving a live connection to the data.

### Endpoint 1: Contest Types

**Endpoint:** `GET https://pokeapi.co/api/v2/contest-type/{id or name}/`

**Route Parameters:** The route parameter is `{id or name}`. It allows the user to get a specific contest type using either its numerical ID (such as `1`, `2`, etc.) or its name (e.g. `cool`, `beautiful`, etc.)

**Data Retrieved:** This endpoint returns information about a Pokémon contest type, including its ID, name, associated berry flavor, and names in differnet langauges. One example is the "cool" contest type has a corresponding ID of `1`, and is associated with the "spicy" berry flavor, and is associated with the color "red". 

### Endpoint 2: Currencies

**Endpoint:** `GET https://pokeapi.co/api/v2/currency/{id or name}/`

**Route Parameters:** The route parameter is `{id or name}`. It allows the user to get a specific currency using either its numerical ID (e.g., `1`) or its name (e.g., `poke-dollar`).

**Data Retrieved:** This endpoint returns information about a currency used in Pokémon games, including its ID, name, and names in different languages. For example, the "poke-dollar" currency has an ID of `1` and is called "Poké Dollar" in English.

In [ ]:
import json
import requests

# Load contest type data
with open("contest_type.json", encoding="utf-8") as handle:
    contest_data = json.load(handle)

print(contest_data.keys())

# Load currency data
with open("currency.json", encoding="utf-8") as handle:
    currency_data = json.load(handle)

print(currency_data.keys())

dict_keys(['id', 'name', 'berry_flavor', 'names'])
dict_keys(['id', 'name', 'names'])


In [2]:
# Contest type
print("Contest Type:", contest_data["name"])
print("ID:", contest_data["id"])
print("Berry Flavor:", contest_data["berry_flavor"]["name"])

# Currency
print("Currency:", currency_data["name"])
print("ID:", currency_data["id"])

Contest Type: tough
ID: 5
Berry Flavor: sour
Currency: poke-dollar
ID: 1


In [ ]:
# API endpoints
# Contest 
CONTEST_ENDPOINT = "https://pokeapi.co/api/v2/contest-type/1/"

contest_response = requests.get(CONTEST_ENDPOINT, timeout=10)
contest_response.raise_for_status()

live_contest_data = contest_response.json()

# Expected keys in the berry_flavor dictionary
BERRY_FLAVOR_KEYS = {"name", "url"}

# Length of the names list from fixture
EXPECTED_LENGTH = len(contest_data["names"])

print(live_contest_data.keys())
print("Contest Type:", live_contest_data["name"])
print("Berry Flavor:", live_contest_data["berry_flavor"]["name"])


# Currency 
CURRENCY_ENDPOINT = "https://pokeapi.co/api/v2/currency/1/"
currency_response = requests.get(CURRENCY_ENDPOINT, timeout=10)
currency_response.raise_for_status()

live_currency_data = currency_response.json()

# Expected number of translated names in fixture
CURRENCY_EXPECTED_LENGTH = len(currency_data["names"])

print(live_currency_data.keys())
print("Currency:", live_currency_data["name"])
print("ID:", live_currency_data["id"])


dict_keys(['id', 'name', 'berry_flavor', 'names'])
Contest Type: cool
Berry Flavor: spicy
dict_keys(['id', 'name', 'names'])
Currency: poke-dollar
ID: 1


In [ ]:
def validate_contest_data(payload):
    # Check that the overall response is a dictionary
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object")

    # Check that berry_flavor exists and is a dictionary
    if "berry_flavor" not in payload or not isinstance(payload["berry_flavor"], dict):
        raise ValueError("Missing berry_flavor object")

    # Compare the expected keys with the actual keys in berry_flavor
    missing = BERRY_FLAVOR_KEYS - set(payload["berry_flavor"])

    # Raise an error if any expected keys are missing
    if missing:
        raise ValueError(f"Missing berry_flavor fields: {sorted(missing)}")

    # Check that names exists and is a list
    if "names" not in payload or not isinstance(payload["names"], list):
        raise ValueError("Missing names list")

    # Find the length of the names list
    length = len(payload["names"])

    # Check if the length matches the saved fixture
    if length != EXPECTED_LENGTH:
        raise ValueError(f"Names list has incorrect length: {length}")

    # Return the payload if all validation checks pass
    return payload

def validate_currency_data(payload):
    # Check that the overall response is a dictionary
    if not isinstance(payload, dict):
        raise TypeError("API payload must be a JSON object")

    # Check that names exists and is a list
    if "names" not in payload or not isinstance(payload["names"], list):
        raise ValueError("Missing currency names list")

    # Find the length of the currency names list
    length = len(payload["names"])

    # Check if the length matches the saved currency fixture
    if length != CURRENCY_EXPECTED_LENGTH:
        raise ValueError(f"Currency names list has incorrect length: {length}")

    # Return the payload if validation passes
    return payload

In [ ]:
# Same structure as the teacher's acquire_weather() example
# Contest Type API
def acquire_contest(use_live=False):
    source = "cached fixture"
    payload = contest_data
    if use_live:
        try:
            # Send a GET request to the API. Stop waiting if the API connection or response takes too long
            response = requests.get(CONTEST_ENDPOINT, timeout=10)
            # Check the HTTP status code. Raise an error for HTTP error statuses (4xx or 5xx)
            # 200 means the request succeeded.
            # 4xx codes indicate client errors, such as 404 Not Found.
            # 5xx codes indicate server errors, such as 500 Internal Server Error.
            # raise_for_status() raises an exception for 4xx and 5xx errors.
            response.raise_for_status()
            # Convert the JSON response to Python data
            payload = response.json()
            # Check that the response has the expected structure
            validate_contest_data(payload)
            source = response.url
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = contest_data
    return validate_contest_data(payload), source

# Currency API: same steps, but use Currency's endpoint, fixture, and validator
def acquire_currency(use_live=False):
    source = "cached fixture"
    payload = currency_data
    if use_live:
        try:
            # Send a GET request; timeout=10 prevents waiting indefinitely for a response
            response = requests.get(CURRENCY_ENDPOINT, timeout=10)
            # Raise an error for HTTP error statuses (4xx or 5xx)
            response.raise_for_status()
            # Convert the JSON response to Python data
            payload = response.json()
            # Check that the response has the expected structure
            validate_currency_data(payload)
            source = response.url
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = currency_data
    return validate_currency_data(payload), source

In [ ]:
# Test both saved fixtures (no internet connection needed)
contest_fixture, contest_fixture_source = acquire_contest(use_live=False)
currency_fixture, currency_fixture_source = acquire_currency(use_live=False)

# Try live API data; use fixtures if a request or validation fails
contest_live, contest_source = acquire_contest(use_live=True)
currency_live, currency_source = acquire_currency(use_live=True)

# Print the data source to know whether live data or the fixture was used
print("Contest source:", contest_source)
print("Currency source:", currency_source)

# Print the validated responses
print("Contest Type:")
print(json.dumps(contest_live, indent=4))
print("Currency:")
print(json.dumps(currency_live, indent=4))

# Access specific values
print("Contest Type:", contest_live["name"])
print("Berry Flavor:", contest_live["berry_flavor"]["name"])
print("Contest Names:", len(contest_live["names"]))
print("Currency:", currency_live["name"])
print("Currency Names:", len(currency_live["names"]))

{
    "id": 1,
    "name": "cool",
    "berry_flavor": {
        "name": "spicy",
        "url": "https://pokeapi.co/api/v2/berry-flavor/1/"
    },
    "names": [
        {
            "name": "\u304b\u3063\u3053\u3088\u3055",
            "color": "",
            "language": {
                "name": "ja-hrkt",
                "url": "https://pokeapi.co/api/v2/language/1/"
            }
        },
        {
            "name": "\u5e25\u6c23",
            "color": "\u7d05",
            "language": {
                "name": "zh-hant",
                "url": "https://pokeapi.co/api/v2/language/4/"
            }
        },
        {
            "name": "Sang-froid",
            "color": "Rouge",
            "language": {
                "name": "fr",
                "url": "https://pokeapi.co/api/v2/language/5/"
            }
        },
        {
            "name": "Cool",
            "color": "Red",
            "language": {
                "name": "en",
                "url": "https://p

### API Validation

I validated portions of two PokéAPI endpoints: Contest Type and Currency.

**Dictionary Validation (Contest Type):** I checked that the overall response and `berry_flavor` are dictionaries using `isinstance()`. I then used set subtraction to compare the expected keys (`name` and `url`) with the actual keys in `berry_flavor`. If any keys are missing, the function raises a `ValueError`.

**List Validation (Currency):** I checked that `names` exists and is a list. I used `len()` to compare its length against the expected length from the saved currency fixture. If the lengths differ, the function raises a `ValueError`.

**Timeout and HTTP Status:** I used `timeout=10` to prevent the program from waiting indefinitely for a response. I used `raise_for_status()` to detect HTTP errors, including 4xx client errors (such as 404 Not Found) and 5xx server errors (such as 500 Internal Server Error). A 200 status indicates a successful request.

source: https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status

**Error Handling:** I used `try` and `except` to catch connection, timeout, HTTP, JSON, and validation errors. If the live request fails or returns data that doesn't match the expected structure, the program falls back to the saved fixture.

### Pokémon Contest Type Keys

Four main keys:

- **`id`**: The number identifying the contest type
- **`name`**: The name of the contest type (e.g., `tough`)
- **`berry_flavor`**: A dictionary containing two keys: name (the berry flavor, sour) and url (a link to the berry flavor's API endpoint)
- **`names`**: A list of dictionaries containing the contest type's names in different languages. Each dictionary has a name, color, and language. The language key contains another dictionary with the language's name and url